In [3]:
import openpyxl
import win32com.client as win32
from datetime import datetime
import calendar
import sys

# Apri il file Excel
file_path = r'C:\Users\Balestra\Desktop\Nuovo Foglio di lavoro di Microsoft Excel.xlsx'  # Assicurati di inserire il percorso corretto
try:
    workbook = openpyxl.load_workbook(file_path)
except FileNotFoundError:
    print(f"Excel file not found at {file_path}")
    sys.exit(1)

# Ottieni il nome del prossimo mese in inglese
current_month = datetime.now().month
next_month = (current_month % 12) + 1
next_month_name = calendar.month_name[next_month]

# Vai allo sheet del prossimo mese
if next_month_name in workbook.sheetnames:
    sheet = workbook[next_month_name]
else:
    print(f"Sheet for {next_month_name} not found.")
    sys.exit(1)

# Leggi gli impegni e le date
impegni = []
for row in sheet.iter_rows(min_row=2, values_only=True):
    impegno = row[0]  # Nome dell'impegno
    if not impegno:
        continue  # Salta le righe vuote
    for idx, cell_value in enumerate(row[1:], start=1):
        if cell_value:  # Se la cella è spuntata
            giorno = idx  # Il giorno corrisponde all'indice della colonna
            impegni.append(f"{impegno} on {next_month_name} {giorno}")

if not impegni:
    print("No commitments found for next month.")
    sys.exit(0)

# Crea la bozza di email in Outlook
try:
    outlook = win32.Dispatch('outlook.application')
    namespace = outlook.GetNamespace("MAPI")
except Exception as e:
    print(f"Error initializing Outlook: {e}")
    sys.exit(1)

# Trova la cartella "Test"
account_email = "davide.balestra.001@student.uni.lu"  # Inserisci il tuo nome account email
try:
    account_folder = namespace.Folders.Item(account_email)
    test_folder = account_folder.Folders("test")
except Exception as e:
    print(f"Error accessing 'Test' folder: {e}")
    sys.exit(1)

mail = outlook.CreateItem(0)  # 0 per olMailItem
mail.Subject = f"Summary of Commitments for {next_month_name}"
mail.Body = "Dear Team,\n\nHere are the commitments for next month:\n\n"
mail.Body += "\n".join(impegni)
mail.Body += "\n\nBest Regards,\nDavide"

# Salva la bozza nella cartella "Test"
try:
    mail.Save()
    mail.Move(test_folder)
except Exception as e:
    print(f"Error saving or moving the email: {e}")
    sys.exit(1)

print("Draft email created and moved to 'Test' folder in Outlook.")


No commitments found for next month.


SystemExit: 0

In [4]:
import openpyxl
import win32com.client as win32
from datetime import datetime
import calendar

# Apri il file Excel
file_path = r'C:\Users\Balestra\Desktop\Nuovo Foglio di lavoro di Microsoft Excel.xlsx'  # Assicurati di inserire il percorso corretto
workbook = openpyxl.load_workbook(file_path)

# Ottieni il nome del prossimo mese
current_month = datetime.now().month
next_month = (current_month % 12) + 1
next_month_name = calendar.month_name[next_month]

# Vai allo sheet del prossimo mese
if next_month_name in workbook.sheetnames:
    sheet = workbook[next_month_name]
else:
    print(f"Sheet for {next_month_name} not found.")
    exit()

# Leggi gli impegni e le date
impegni = []
for row in sheet.iter_rows(min_row=2, max_col=sheet.max_column, values_only=False):
    impegno = row[0].value  # Nome dell'impegno
    for idx, cell in enumerate(row[1:], start=1):
        if cell.value == True:  # Se è spuntato
            giorno = idx  # Il giorno è la colonna corrispondente
            data_evento = f"{next_month_name} {giorno}"
            impegni.append((data_evento, impegno))

# Ordina gli impegni per data
impegni.sort(key=lambda x: int(x[0].split()[1]))

# Crea la bozza di email in Outlook
outlook = win32.Dispatch('outlook.application')
namespace = outlook.GetNamespace("MAPI")

account_email = "davide.balestra.001@student.uni.lu"  # Inserisci il tuo nome account email
# Trova la cartella "Test"
folder = namespace.Folders.Item(account_email).Folders("Test")

mail = outlook.CreateItem(0)
mail.Subject = f"Summary of Commitments for {next_month_name}"
mail.Body = "Dear Team,\n\nHere are the commitments for next month:\n\n"

# Aggiungi la lista di impegni in ordine cronologico
for data_evento, impegno in impegni:
    mail.Body += f"• {data_evento}: **{impegno}**\n"

mail.Body += "\n\nBest Regards,\nDavide"

# Imposta i destinatari e salva la bozza
mail.To = "bal6.h@gmail.com"
mail.CC = "gds@gmail.com"

# Salva la bozza nella cartella "Test"
mail.Save()
mail.Move(folder)

print("Draft email created and moved to 'Test' folder in Outlook.")


Draft email created and moved to 'Test' folder in Outlook.


In [10]:
import pandas as pd
import datetime
import win32com.client as win32
import numpy as np

# Get next month
today = datetime.date.today()
first_day_next_month = datetime.date(today.year + today.month // 12, today.month % 12 + 1, 1)
# Month names in English
month_names_english = ['January', 'February', 'March', 'April', 'May', 'June',
                       'July', 'August', 'September', 'October', 'November', 'December']
next_month_name = month_names_english[first_day_next_month.month - 1]

file_path = r'C:\Users\Balestra\Desktop\Nuovo Foglio di lavoro di Microsoft Excel.xlsx'
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

if next_month_name in sheet_names:
    df = pd.read_excel(xls, sheet_name=next_month_name, header=None)
else:
    print(f"Sheet for month {next_month_name} not found.")
    exit()

# Identify the row containing day numbers
day_numbers_list = list(range(1, 32))
day_row_index = None
for i in range(len(df)):
    row_values = df.iloc[i].dropna().values
    # Check if the row contains at least 5 day numbers
    day_numbers_in_row = [val for val in row_values if val in day_numbers_list]
    if len(day_numbers_in_row) >= 5:
        day_row_index = i
        break

if day_row_index is None:
    print("Could not find the row with day numbers.")
    exit()

# Identify the column containing event names
event_col_index = None
for col in df.columns:
    col_values = df[col].iloc[day_row_index + 1:].dropna()
    if col_values.dtype == 'object':
        # Assume event names are strings
        if col_values.apply(lambda x: isinstance(x, str)).sum() > len(col_values) / 2:
            event_col_index = col
            break

if event_col_index is None:
    print("Could not find the column with event names.")
    exit()

# Extract day numbers and their corresponding columns
day_numbers_row = df.iloc[day_row_index]
day_columns = []
day_numbers_clean = []
for idx, val in enumerate(day_numbers_row.values):
    if pd.notnull(val) and val in day_numbers_list:
        day_columns.append(idx)
        day_numbers_clean.append(int(val))

# Extract event names and their corresponding rows
event_names_series = df.iloc[day_row_index + 1:, event_col_index].dropna()
event_rows = event_names_series.index
event_names = event_names_series.values

# Collect scheduled events
scheduled_events = []
for idx, event_name in zip(event_rows, event_names):
    event_schedule = []
    for col_idx, day_number in zip(day_columns, day_numbers_clean):
        cell_value = df.iloc[idx, col_idx]
        if pd.notnull(cell_value):
            event_schedule.append(day_number)
    if event_schedule:
        scheduled_events.append({'name': event_name, 'dates': event_schedule})

# Prepare the email body
email_body = '<p>Dear Team,</p>'
email_body += '<p>Please find below the commitments for next month:</p>'
email_body += '<ul>'
for event in scheduled_events:
    event_name = event['name']
    dates = event['dates']
    for day in dates:
        # Format the date
        event_date = datetime.date(first_day_next_month.year, first_day_next_month.month, day)
        date_str = event_date.strftime('%d/%m')
        # Create the list item
        list_item = '<li><b>{}</b>: <span style="background-color: yellow;">{}</span></li>'.format(event_name, date_str)
        email_body += list_item
email_body += '</ul>'
email_body += '<p>Best regards,<br>Davide Balestra</p>'

# Create the email
outlook = win32.Dispatch('Outlook.Application')
mail = outlook.CreateItem(0)  # 0: olMailItem

# Set the recipients
mail.To = "bal6.h@gmail.com"
mail.CC = "gds@gmail.com"

# Set the subject
mail.Subject = f"Upcoming Events for {next_month_name} {first_day_next_month.year}"

# Set the account
account_email = "davide.balestra.001@student.uni.lu"

accounts = outlook.Session.Accounts
for account in accounts:
    if account.SmtpAddress == account_email:
        mail._oleobj_.Invoke(*(64209, 0, 8, 0, account))
        break

# Set the email body
mail.HTMLBody = email_body

# Save the draft in 'Test' folder
namespace = outlook.GetNamespace("MAPI")
# Get default Inbox folder
inbox = namespace.GetDefaultFolder(6)  # 6: Inbox

# Now, get the 'Test' folder. Assuming 'Test' is a subfolder of Inbox.
test_folder = None
for folder in inbox.Folders:
    if folder.Name == 'Test':
        test_folder = folder
        break

if test_folder is None:
    # Create 'Test' folder if it doesn't exist
    test_folder = inbox.Folders.Add('Test')

# Save the email to the 'Test' folder
mail.Save()  # Save the email first
mail.Move(test_folder)

print("Email draft has been created and saved in the 'Test' folder.")


Email draft has been created and saved in the 'Test' folder.


In [ ]:
codice per calendario 

In [12]:
import win32com.client as win32
import datetime
import re

# Get the first day of the next month
today = datetime.date.today()
first_day_next_month = datetime.date(today.year + today.month // 12, today.month % 12 + 1, 1)
next_month = first_day_next_month.month

# Set up Outlook
outlook = win32.Dispatch('Outlook.Application')
namespace = outlook.GetNamespace("MAPI")

# Navigate to the "Test" folder
inbox = namespace.GetDefaultFolder(6)  # 6: Inbox
test_folder = None
for folder in inbox.Folders:
    if folder.Name == 'Test':
        test_folder = folder
        break

if not test_folder:
    print("Test folder not found.")
    exit()

# Get the latest email from the "Test" folder
items = test_folder.Items
items.Sort("[ReceivedTime]", True)  # Sort by ReceivedTime in descending order
latest_email = items.GetFirst()

if not latest_email:
    print("No emails found in the Test folder.")
    exit()

# Extract event details from the email body
email_body = latest_email.HTMLBody

# Regular expression to match events: "<b>EventName</b>: <span style="...">DD/MM</span>"
event_pattern = re.compile(r'<b>(.*?)</b>:\s*<span.*?>(\d{2}/\d{2})</span>')

matches = event_pattern.findall(email_body)
events = []

# Convert extracted dates to actual datetime objects
for event_name, event_date in matches:
    try:
        day, month = map(int, event_date.split("/"))
        event_datetime = datetime.date(first_day_next_month.year, month, day)

        # Only add events if the month matches the next month
        if event_datetime.month == next_month:
            events.append({"name": event_name, "date": event_datetime})
    except ValueError:
        print(f"Failed to parse date: {event_date}")

if not events:
    print("No valid events found for the next month.")
    exit()

# Add the events to the Outlook Calendar
calendar = namespace.GetDefaultFolder(9)  # 9: Calendar
for event in events:
    appointment = outlook.CreateItem(1)  # 1: olAppointmentItem
    appointment.Subject = event["name"]
    appointment.Start = event["date"].strftime("%Y-%m-%d")
    appointment.AllDayEvent = True  # Set as an all-day event
    appointment.Save()

print("Events successfully added to the Outlook Calendar.")

# Prepare the confirmation email body
email_body_confirm = '<p>Dear Team,</p>'
email_body_confirm += '<p>The following events have been successfully added to the Outlook calendar for next month:</p>'
email_body_confirm += '<ul>'
for event in events:
    event_name = event["name"]
    event_date = event["date"].strftime("%d/%m")
    list_item = f'<li><b>{event_name}</b>: {event_date}</li>'
    email_body_confirm += list_item
email_body_confirm += '</ul>'
email_body_confirm += '<p>Best regards,<br>Your Name</p>'

# Create a new draft email
confirm_mail = outlook.CreateItem(0)  # 0: olMailItem

# Set the recipients for the confirmation email
confirm_mail.To = "bal6.h@gmail.com"
confirm_mail.CC = "gds@gmail.com"

# Set the subject for the confirmation email
confirm_mail.Subject = f"Confirmation of Events Added for {first_day_next_month.strftime('%B %Y')}"

# Set the email body
confirm_mail.HTMLBody = email_body_confirm

# Save the draft in the 'Test' folder
confirm_mail.Save()  # Save the email first
confirm_mail.Move(test_folder)

print("Confirmation email draft has been created and saved in the 'Test' folder.")


Events successfully added to the Outlook Calendar.
Confirmation email draft has been created and saved in the 'Test' folder.
